# Lesson 12 Lab — Reduction and Scan

**Puzzle:** When tree reduction, prefix dependency, and neutral values change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates tree reduction, prefix dependency, and neutral values and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Reduction collapses a row to one value; scan produces every prefix. Triton expresses both as blocked tensor operations, but their dependency graphs and output traffic differ. Tail padding must still use the correct identity.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["tree reduction, prefix dependency, and neutral values"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

A very large row kept in one program can increase register pressure and reduce active work even when the source is concise.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 12
LESSON_TITLE = 'Reduction and Scan'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260825
}


## 5. Freeze the experiment

**Experiment:** Run row sum and inclusive cumsum on identical 4,096 by 256 tensors and verify both outputs.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.0161920003592968,
  "secondary": 0.015471999999135733,
  "max_abs_error": 1.52587890625e-05,
  "passed": true,
  "details": {
    "reduction_samples_ms": [
      0.031808000057935715,
      0.022207999601960182,
      0.0191040001809597,
      0.018079999834299088,
      0.01945599913597107,
      0.017855999991297722,
      0.0161920003592968,
      0.015456000342965126,
      0.01568000018596649,
      0.015200000256299973,
      0.01600000075995922,
      0.01539199985563755,
      0.016127999871969223,
      0.017696000635623932,
      0.015424000099301338,
      0.015647999942302704,
      0.01539199985563755,
      0.0161920003592968,
      0.020927999168634415,
      0.017216000705957413
    ],
    "scan_samples_ms": [
      0.017152000218629837,
      0.016127999871969223,
      0.015904000028967857,
      0.015072000212967396,
      0.016224000602960587,
      0.01571200042963028,
      0.015744000673294067,
      0.015936000272631645,
      0.0154560003429651

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Reduction median | 0.0162 ms |
| Scan median | 0.0155 ms |
| Maximum absolute error | 1.526e-05 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The row reduction took 0.0162 ms and inclusive scan 0.0155 ms. Their communication structures differ even with identical input shapes.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Select a one-program or multi-stage algorithm from row width and resource evidence, not API brevity.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 12,
  "title": "Reduction and Scan",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260825
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.0161920003592968,
    "secondary": 0.015471999999135733,
    "max_abs_error": 1.52587890625e-05,
    "passed": true,
    "details": {
      "reduction_samples_ms": [
        0.031808000057935715,
        0.022207999601960182,
        0.0191040001809597,
        0.018079999834299088,
        0.01945599913597107,
        0.017855999991297722,
        0.0161920003592968,
        0.015456000342965126,
        0.01568000018596649,
        0.015200000256299973,
        0.01600000075995922,
        0.01539199985563755,
        0.016127999871969223,
        0.017696000635623932,
      

## 10. Make the bounded decision

> Select a one-program or multi-stage algorithm from row width and resource evidence, not API brevity.

**Failure analysis:** A very large row kept in one program can increase register pressure and reduce active work even when the source is concise.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
